In [0]:
%pip install loguru

In [0]:
import pandas as pd
import numpy as np
from loguru import logger
import os
from datetime import datetime
import random

In [0]:
csv_path = '/Workspace/Users/pundpaltejas@gmail.com/DE-Practice/Python/Pandas/cust_details.csv'
df = pd.read_csv(csv_path)
print(df)

In [0]:

num_new_rows = 28
random.seed(42)  # for reproducibility
np.random.seed(42)

# ---------- SOURCE POOLS ----------
first_names = [
    "Aarav", "Vivaan", "Aditya", "Kabir", "Arjun", "Ishaan", "Rohan", "Atharv",
    "Kunal", "Pranav", "Rahul", "Sarthak", "Prateek", "Tejas", "Atul", "Siddharth",
    "Saurabh", "Nikhil", "Yash", "Harsh", "Om", "Shreyas", "Ritvik", "Aman",
    "Vivek", "Aakash", "Manish", "Aniket", "Mayur", "Abhishek", "Anirudh", "Rajat",
]

last_names = [
    "Patil", "Sharma", "Verma", "Jadhav", "Kulkarni", "Gupta", "Joshi", "Chavan",
    "Kumar", "Mehta", "Deshmukh", "Yadav", "Singh", "Bhosale", "Sawant", "Pawar",
    "Naik", "Gawande", "Reddy", "Iyer", "Pandey", "Thakur", "Shinde", "Salunkhe",
]

email_domains = ["gmail.com", "yahoo.com", "outlook.com", "hotmail.com", "proton.me"]

# Helper to create a realistic name
def make_name():
    return f"{random.choice(first_names)} {random.choice(last_names)}"

# Helper to create email from name
def make_email(name):
    local = name.lower().replace(" ", ".")
    return f"{local}{random.randint(1, 999)}@{random.choice(email_domains)}"

# ---------- GENERATE BASE NEW ROWS ----------
new_rows = []

# Choose a starting cust_id higher than existing max to avoid accidental clashes
start_id = int(df["cust_id"].max()) + 1 if not df.empty else 1001

for i in range(num_new_rows):
    name = make_name()
    email = make_email(name)
    row = {
        "cust_id": start_id + i,  # sequential ids
        "cust_name": name,
        "cust_email": email,
    }
    new_rows.append(row)

df_new = pd.DataFrame(new_rows)

# ---------- INTRODUCE VARIETY: DUPLICATES ----------
# 1) Duplicate cust_email across 4 different rows
dup_email = df_new.loc[0, "cust_email"]
dup_indices_email = np.random.choice(df_new.index[1:], size=3, replace=False)
for idx in dup_indices_email:
    df_new.at[idx, "cust_email"] = dup_email

# 2) Duplicate cust_name across 3 different rows
dup_name = df_new.loc[1, "cust_name"]
dup_indices_name = np.random.choice(df_new.index[2:], size=2, replace=False)
for idx in dup_indices_name:
    df_new.at[idx, "cust_name"] = dup_name

# (Optional) 3) Duplicate a cust_id to create ID clash (useful for dedup practice)
# We'll duplicate exactly 2 cust_id values
dup_indices_id = np.random.choice(df_new.index[3:], size=2, replace=False)
for idx in dup_indices_id:
    df_new.at[idx, "cust_id"] = df_new.loc[idx - 1, "cust_id"]  # set equal to previous id

# ---------- INTRODUCE NULLS / BLANKS ----------
# Make 2 rows have nulls in cust_email
null_email_indices = np.random.choice(df_new.index, size=2, replace=False)
df_new.loc[null_email_indices, "cust_email"] = np.nan

# Make 2 rows have blank strings in cust_name
blank_name_indices = np.random.choice(df_new.index.difference(null_email_indices), size=2, replace=False)
df_new.loc[blank_name_indices, "cust_name"] = ""

# Make 1 row have null in cust_name and 1 row have null in cust_id
remaining_indices = df_new.index.difference(np.concatenate([null_email_indices, blank_name_indices]))
if len(remaining_indices) >= 2:
    df_new.loc[remaining_indices[:1], "cust_name"] = np.nan
    # For cust_id null, we choose a different row
    df_new.loc[remaining_indices[1:2], "cust_id"] = np.nan

# ---------- CONCAT & SAVE ----------
df_final = pd.concat([df, df_new], ignore_index=True)
df_final.to_csv(csv_path, index=False)

# ---------- SUMMARY ----------
print("New rows added:", num_new_rows)
print("\nDuplicate summary on appended data:")
print("  Duplicate cust_email count (including NaNs ignored):",
      df_final["cust_email"].duplicated(keep=False).sum())

print("  Duplicate cust_name count (including blanks/NaNs ignored):",
      df_final["cust_name"].duplicated(keep=False).sum())

print("\nNull/blank summary:")
print("  Null cust_id:", df_final["cust_id"].isna().sum())
print("  Null cust_name:", df_final["cust_name"].isna().sum())
print("  Blank cust_name:", (df_final["cust_name"] == "").sum())
print("  Null cust_email:", df_final["cust_email"].isna().sum())

# Optional: peek the tail to see a sample
print("\nTail preview:")
print(df_final.tail(10))

In [0]:
print(df_final)

In [0]:
df_final_shape = df_final.shape
print(df_final_shape)

In [0]:
filtered_tejas_df = df_final.loc[df_final['cust_name'].str.contains('Tejas',na=False),['cust_name','cust_email']]
filtered_tejas_df_count = filtered_tejas_df.shape[0]
print(filtered_tejas_df_count)
print(filtered_tejas_df)

In [0]:
tejas_thakur_count = filtered_tejas_df.loc[filtered_tejas_df['cust_name'] == 'Tejas Thakur'].shape[0]
print(tejas_thakur_count)

In [0]:
tejas_count = filtered_tejas_df[filtered_tejas_df['cust_name'] == 'Tejas']['cust_name'].count()
print(tejas_count)

In [0]:
df_final = df_final.drop_duplicates(subset = ["cust_id","cust_email","cust_name"])
df_final = df_final.dropna(subset = ["cust_id"])
df_final = df_final.dropna(subset = ["cust_email"])
df_final = df_final.dropna(subset = ["cust_name"])
df_final.reset_index(drop=True, inplace=True)
# print(df_final)

df_final_count = df_final.shape[0]
print(f"cleaned data df_final count : {df_final_count}")

df_final.to_csv('/Workspace/Users/pundpaltejas@gmail.com/DE-Practice/Python/Pandas/cust_details_cleaned.csv', index=False)

In [0]:
#Reading cleaned csv
df_final = pd.read_csv('/Workspace/Users/pundpaltejas@gmail.com/DE-Practice/Python/Pandas/cust_details_cleaned.csv')
df_final

In [0]:
# replacing blank with NaN
cols = ['cust_name', 'cust_email', 'cust_id']
df_final[cols] = df_final[cols].replace('^\\s*$', np.nan, regex=True)
# df_final
df_final.dropna(subset=['cust_id','cust_name','cust_email'],inplace=True)
df_final

In [0]:
df_final = df_final.rename(columns={'cust_id':'Cust_ID','cust_name':'Cust_Name','cust_email':'Cust_Email'})
df_final.info()

In [0]:
df_final = df_final.astype({'Cust_ID': 'int','Cust_Name': 'str','Cust_Email': 'str'})
df_final.info()

In [0]:
#Fill nan values with 0
#Actually now there are no nan values it's just for learn purpose
df_final[['Cust_ID','Cust_Name','Cust_Email']] = df_final[['Cust_ID','Cust_Name','Cust_Email']].fillna(0)

In [0]:
# Display DataFrame info and summary statistics
print("DataFrame Info:")
df_final.info()
print("\nSummary Statistics:")
display(df_final.describe())

In [0]:
# Show value counts for Cust_Name and Cust_Email
print("\nTop 10 most common customer names:")
print(df_final['Cust_Name'].value_counts().head(10))
print("\nTop 10 most common customer emails:")
print(df_final['Cust_Email'].value_counts().head(10))

**Step 1: Summary statistics and info**

- `df_final.info()` shows column types and missing values.
- `df_final.describe()` provides summary statistics for numeric columns.
- `value_counts()` helps identify the most frequent names and emails.